# bn-weight-bias-init-pattern — worked example 2: Check sampled gamma matches N(1, 0.02)

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `bn-weight-bias-init-pattern`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import matplotlib.pyplot as plt

## Concept

Because BN gamma is drawn from `N(1, 0.02)`, a wide BN layer's weight vector should have a sample mean very close to 1.0 and a sample standard deviation very close to 0.02. The bias is deterministic — exactly zero — so its mean and std are both 0.

## Worked solution

**Step 1 — build a single wide BN layer.** A large `num_features` (here 4096) gives a big sample, so the empirical statistics of gamma will closely track the `N(1, 0.02)` population values. With few channels the sample mean would be noisy.

**Step 2 — apply the DCGAN init inline.** `nn.init.normal_(layer.weight, 1.0, 0.02)` fills gamma; `nn.init.zeros_(layer.bias)` fills beta. We re-seed with `t.manual_seed(0)` first so the draw is reproducible — the grader does not reseed for us.

**Step 3 — measure.** `layer.weight.mean()` should be ~1.0 and `layer.weight.std()` ~0.02. The unbiased sample std of 4096 draws from `N(1, 0.02)` lands within a couple percent of 0.02. `layer.bias` is identically zero, so its mean is 0 and its max-abs is 0.

**Step 4 — why it matters.** This confirms the init is a *near-identity affine map*: gamma ≈ 1 means the BN barely rescales, and beta = 0 means no shift. That is exactly the well-conditioned starting point DCGAN wants before training perturbs it.

In [ ]:
import torch.nn as nn

t.manual_seed(0)

layer = nn.BatchNorm1d(4096)
nn.init.normal_(layer.weight, 1.0, 0.02)
nn.init.zeros_(layer.bias)

gamma_mean = layer.weight.mean().item()
gamma_std = layer.weight.std().item()
beta_absmax = layer.bias.abs().max().item()

print("gamma mean (~1.0):", round(gamma_mean, 4))
print("gamma std  (~0.02):", round(gamma_std, 4))
print("beta max-abs (==0):", beta_absmax)